## Creación de un *CRUD* con `sqlite3` (intermedio)
En esta versión del código, haré uso de funciones para simplificar las tareas del CRUD.

Recomendación: instalar extensión [SQLite Viewer](https://marketplace.visualstudio.com/items?itemName=qwtel.sqlite-viewer)

In [2]:
import sqlite3
from pathlib import Path
import os

### Listar archivos de base de datos

In [2]:
os.getcwd()

'/home/daniel/code/python_redcross/sqlite'

In [6]:
db_files = [i for i in Path("./databases/").iterdir() if i.suffix == ".db"]
print(db_files)

[PosixPath('databases/library.db')]


In [7]:
def init_connection(db_file : str) -> tuple[sqlite3.Connection, sqlite3.Cursor] | None :
    # check if file exists
    if (Path("./databases/") / db_file).exists():
        try:
            conn = sqlite3.connect(db_file)
            cursor = conn.cursor()
            print("Connection created successfuly!")
            return conn, cursor
        except Exception as e:
            print(f"Error: {e}")
            return None
    else:
        print(f"{db_file} does not exist. I will create it.")
        try:
            conn = sqlite3.connect(db_file)
            cursor = conn.cursor()
            print("Connection created successfuly!")
            return conn, cursor
        except Exception as e:
            print(f"Error: {e}")
            return None

In [8]:
conn, cursor = init_connection("library.db")

Connection created successfuly!


### Funciones auxiliares

Crearé algunas funciones auxiliares que me permitan realizar algunas tareas que ayuden en las tareas de mantenimiento de la base de datos y sus tablas.

#### Listado de tablas de la BD
Esta función devuelve una lista de las tablas que contiene la BD de la conexión activa.

In [14]:
def list_tables(conn = conn, cursor=cursor):
    tables = (cursor
              .execute("SELECT name FROM sqlite_master WHERE type='table'")
              .fetchall()
    )
    tables = [i[0] for i in tables if i[0] != "sqlite_sequence"]
    return tables

#### Creación de tabla
Esta función crea una tabla a partir de una *query* con el código SQL de creación de la tabla.

In [19]:
def create_table(table_name, query, conn, cursor):
    try:
        # check if table exists
        if not table_name in list_tables():
            cursor.execute(query)
            conn.commit()
            print(f"Table {table_name} was successfuly created!")
        # table was already created
        else:
            print(f"Table {table_name} already exists. No actions were taken.")
    except Exception as e:
        print(f"Error: {e}")

### Creación BD
Creación de la base de datos

In [29]:
create_table_query = '''
create table if not exists books
(id integer primary key autoincrement,
title text unique,
author text,
publish_year integer,
available bool)
'''

In [20]:
create_table("books", create_table_query, conn, cursor)

Table books already exists. No actions were taken.


### Inserción de datos
Insertamos registros en la base de datos

In [25]:
books = [
    ("The Pragmatic Programmer", "Andrew Hunt & David Thomas", 1999, True),
    ("Clean Code", "Robert C. Martin", 2008, True),
    ("Design Patterns", "Gang of Four", 1994, False),
    ("The Mythical Man-Month", "Frederick P. Brooks Jr.", 1975, True),
    ("Introduction to Algorithms", "Cormen, Leiserson, Rivest & Stein", 1990, False),
    ("You Don't Know JS", "Kyle Simpson", 2015, True),
    ("The Art of Computer Programming", "Donald E. Knuth", 1968, False),
    ("Refactoring", "Martin Fowler", 1999, True),
    ("Structure and Interpretation of Computer Programs", "Abelson & Sussman", 1984, True),
    ("Code Complete", "Steve McConnell", 1993, False),
]

Inserción de registros usando *placeholders* (?)

In [27]:
insert_query_placeholder = '''
insert into books (title, author, published_year, available)
values (?, ?, ?, ?)
'''

In [28]:
for b in books:
    cursor.execute(insert_query_placeholder, b)

In [29]:
conn.commit()

Inserción de registros usando *placeholders* (:column_name)

In [27]:
def insert_records(table_name, cols, values_tuples, conn=conn, cursor=cursor):
    # create a dictionary containing column names as keys
    # and values contained in each tuple provided in values
    values_dict = [{k:v for k,v in zip(cols, t)}
                   for t in values_tuples]
    
    # create placeholders
    placeholders = []
    for c in cols:
        placeholders.append(f":{c}")
    placeholders = ", ".join(placeholders)

    insert_query = f'''
    insert into {table_name} ({", ".join(cols)})
    values ({placeholders}) ;
    '''

    cursor.execute(insert_query, values_dict)

    print(insert_query)

In [28]:
insert_records("books", ["title", "author", "publish_year", "available"], books)

OperationalError: table books has no column named publish_year

In [35]:
cols = ["title", "author", "published_year", "available"]
books_dict = [{k:v for k,v in zip(cols, t)} for t in books]

In [36]:
books_dict[0]

{'title': 'The Pragmatic Programmer',
 'author': 'Andrew Hunt & David Thomas',
 'published_year': 1999,
 'available': True}

In [7]:
insert_query = '''
insert into books (title, author, published_year, available)
values (:title, :author, :published_year, :available)
'''

In [11]:
cursor.executemany(insert_query, books)

In [22]:
conn.commit()

Read the table

In [5]:
read_query = "select * from books"

In [11]:
try:
    records = conn.execute(read_query).fetchall()
    print(*records, sep="\n")

except sqlite3.OperationalError:
    print("The table does not exist!")

(1, 'The Pragmatic Programmer', 'Andrew Hunt & David Thomas', 1999, 1)
(2, 'Clean Code', 'Robert C. Martin', 2008, 1)
(3, 'Design Patterns', 'Gang of Four', 1994, 0)
(4, 'The Mythical Man-Month', 'Frederick P. Brooks Jr.', 1975, 1)
(5, 'Introduction to Algorithms', 'Cormen, Leiserson, Rivest & Stein', 1990, 0)
(6, "You Don't Know JS", 'Kyle Simpson', 2015, 1)
(7, 'The Art of Computer Programming', 'Donald E. Knuth', 2018, 0)
(8, 'Refactoring', 'Martin Fowler', 1999, 1)
(9, 'Structure and Interpretation of Computer Programs', 'Abelson & Sussman', 1984, 1)
(10, 'Code Complete', 'Steve McConnell', 1993, 0)


In [ ]:
# del(records)

In [24]:
for book in records:
    if book[-1]:
        print(f"Available to be rented: \"{book[1]}\".")

### Vaciado de la tabla

In [19]:
# empty the table
cursor.execute("delete from books ;")

# reset the id values
cursor.execute('delete from sqlite_sequence where name ="books" ;')


### Eliminación de la tabla

In [43]:
cursor.execute("drop table books")

Consultamos lista de tablas en la BD

In [28]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
db_list = cursor.fetchall()

In [31]:
for db in db_list:
    if db[0] == "sqlite_sequence":
        continue
    print(f"Database name: {db[0]}")

Database name: books


### Actualización de registros

In [7]:
print(*records, sep="\n")

(1, 'The Pragmatic Programmer', 'Andrew Hunt & David Thomas', 1999, 1)
(2, 'Clean Code', 'Robert C. Martin', 2008, 1)
(3, 'Design Patterns', 'Gang of Four', 1994, 0)
(4, 'The Mythical Man-Month', 'Frederick P. Brooks Jr.', 1975, 1)
(5, 'Introduction to Algorithms', 'Cormen, Leiserson, Rivest & Stein', 1990, 0)
(6, "You Don't Know JS", 'Kyle Simpson', 2015, 1)
(7, 'The Art of Computer Programming', 'Donald E. Knuth', 1968, 0)
(8, 'Refactoring', 'Martin Fowler', 1999, 1)
(9, 'Structure and Interpretation of Computer Programs', 'Abelson & Sussman', 1984, 1)
(10, 'Code Complete', 'Steve McConnell', 1993, 0)


In [8]:
update_query = '''
update books
    set published_year = 2018
    where id = 7
'''

In [9]:
cursor.execute(update_query)

In [10]:
conn.commit()

### Eliminación de registros
Eliminar registros específicos

In [12]:
delete_query = '''
delete from books
where id = 7
'''

In [13]:
cursor.execute(delete_query)

In [14]:
conn.commit()